#1. Basic Tasks 

##1. Create a DataFrame from an in-memory Python list/dict and display it. 

In [0]:

data = [
    {"id": 1, "name": "p1", "age": 28, "city": "New York"},
    {"id": 2, "name": "p2", "age": 35, "city": "San Francisco"},
    {"id": 3, "name": "p3", "age": 42, "city": "Seattle"},
    {"id": 4, "name": "p4", "age": 31, "city": "Boston"},
    {"id": 5, "name": "p5", "age": 29, "city": "Austin"}
]
df = spark.createDataFrame(data)
display(df)

##2. Read a CSV with header=True and inferSchema, then read the same file with an explicit StructType schema; compare the two resulting schemas

In [0]:
df1=spark.read.csv('/Volumes/dev/demo/raw/sales.csv',inferSchema=True,header=True)
df1.printSchema()

In [0]:
from pyspark.sql.types import StructField, StructType, IntegerType, DoubleType,DateType


In [0]:
customSchema = StructType([
    StructField("order_id", IntegerType(), True),        
    StructField("customer_id", IntegerType(), True),
    StructField("transaction_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("discount_amount", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("order_date", DateType(), True),
])
df2 = spark.read.load('/Volumes/dev/demo/raw/sales.csv', format="csv", header="true", schema=customSchema)
df1.display()
df2.display()


In [0]:
print("DF1 Metadata:", df1.schema["order_date"].metadata)
print("DF2 Metadata:", df2.schema["order_date"].metadata)


##3. Apply one filter transformation followed by one action (count() or show()), and explain in your own words why nothing ran until the action.

In [0]:

filtered_df = df1.filter(df1.total_amount > 100)
print(f"Number of orders with total_amount > 100: {filtered_df}")
count_result = filtered_df.count()
print(f"Number of orders with total_amount > 100: {count_result}")

**`Explanation:`**

Nothing ran until count() was called because Spark uses `lazy evaluation`.
When we called filter(), Spark didn't actually process any data - it just 
recorded the transformation in a logical execution plan (a DAG - Directed Acyclic Graph). Only when we invoked count(), which is an `action`, did 
Spark trigger the actual computation. At that moment, Spark optimized the 
entire plan and executed both the read from CSV and the filter together in 
one efficient pass. This lazy approach allows Spark to optimize the entire 
pipeline before execution, avoiding unnecessary intermediate materializations 
and enabling better performance.

#2. Intermediate Tasks

##4. Build a small end-to-end ELT: read CSV, filter, add a column (e.g., ingestion_date), and write the result as Delta.

In [0]:
from pyspark.sql.functions import *
df1=spark.read.csv('/Volumes/dev/demo/raw/sales.csv',inferSchema=True,header=True)
df1 = df1.filter('total_amount > 100').withColumn('ingestion_date', current_timestamp())
df1.write.format("delta").mode("overwrite").saveAsTable("cyntexa_dev.sales.sales_filtered")

##5. Read a JSON file with nested structure and flatten at least one nested field using dot notation or explode()

In [0]:
from pyspark.sql import functions as f
df1=spark.read.json('/Volumes/cyntexa_dev/sales/external_data/orders.json',multiLine=True)
df1.display()
df2=df1.select(f.col("address.city").alias("city"), f.col("address.state").alias("state"),f.col("customer_id"),f.col("name"), f.explode('orders').alias("orders"))
df2.display()

##6. Call .explain() on a multi-step transformation chain and identify, from the physical plan, which steps got pipelined together versus which required a shuffle. 

In [0]:
# Create the transformed DataFrame for explanation
df2.explain()


All of the operations were executed in a single pipeline without any shuffle:
- **PhotonJsonScan**: Reading the JSON file from `/Volumes/cyntexa_dev/sales/external_data/orders.json`
- **PhotonFilter**: Filtering rows where orders array is not null and has size > 0
- **PhotonProject**: Extracting nested fields `address.city` and `address.state`
- **PhotonGenerate**: Exploding the `orders` array to create one row per order
- **PhotonProject**: Final projection to select and alias the desired columns

**Why No Shuffle Was Required:**
No shuffle (data exchange) occurred because:
1. All operations are **narrow transformations** - 
2. There are no wide transformations like `groupBy`, `join`, `repartition`, or `orderBy` that would require redistributing data across partitions


#3. Advanced Tasks 

##7. Take a pandas-based script (your own, or a sample provided by your instructor) and rewrite it in PySpark, documenting at least 3 places where the pandas approach would not scale and how Spark's approach solves it. 

In [0]:
#pandas code
import pandas
import pyspark.pandas as ps
df=ps.read_csv("/Volumes/dev/demo/raw/sales.csv")
df.display()

# PySpark rewrite of pandas code
from pyspark.sql import functions as F
df = spark.read.csv("/Volumes/dev/demo/raw/sales.csv",header=True,inferSchema=True)
df.display()


1. **`MEMORY LIMITATIONS`** - Pandas loads entire dataset into memory on a single machine
 
- Problem: pandas.read_csv() fails when file size exceeds available RAM
- Solution: PySpark distributes data across cluster partitions, processing datasets larger than any single machine's memory

2. **`LAZY EVALUATION`** - Pandas executes operations immediately
- Problem: pandas executes each transformation step-by-step, creating
intermediate copies and wasting resources on unused computations
- Solution: PySpark builds an execution plan (DAG) and optimizes before execution, eliminating unnecessary intermediate steps

3. **`PARALLEL PROCESSING`** - Pandas uses single-threaded execution
- Problem: pandas processes data sequentially on one CPU core, even on multi-core machines or large datasets
- Solution: PySpark automatically parallelizes operations across all cluster cores, reducing processing time proportionally to cluster size

##8. Design a partitioning/write strategy (partitionBy, target file sizes) for a table that will mostly be queried by date range, and justify it using what you know about lazy evaluation and the physical plan. 

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
import random

# Generate sample data for demonstration
start_date = datetime(2024, 1, 1)
data = []
for i in range(10000):
    order_date = start_date + timedelta(days=random.randint(0, 365))
    data.append({
        "order_id": i,
        "order_date": order_date.date(),
        "customer_id": random.randint(1, 1000),
        "product_id": random.randint(1, 100),
        "amount": round(random.uniform(10, 1000), 2)
    })

df = spark.createDataFrame(data)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("order_date") \
    .option("maxRecordsPerFile", 10000) \
    .option("dataChange", "true") \
    .saveAsTable("cyntexa_dev.sales.orders_partitioned")
df.display()

query_df = spark.table("cyntexa_dev.sales.orders_partitioned") \
    .filter(F.col("order_date").between("2024-06-01", "2024-06-30"))

query_df.explain()




In [0]:
query_df.display()

##9. (Data Analyst) Using a Spark DataFrame (not SQL), reproduce a report you'd normally build in Excel/pandas — e.g., monthly revenue by category — and export the result for a dashboard. 

In [0]:
from pyspark.sql import functions as F
df = spark.table("cyntexa_dev.sales.sales_filtered")
df.display()
monthly_revenue = df.groupBy("order_date") \
    .agg(
        F.sum("total_amount").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.avg("total_amount").alias("avg_order_value"),
        F.sum("discount_amount").alias("total_discount")
    ) \
    .orderBy("order_date")
monthly_revenue.display()

monthly_revenue.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cyntexa_dev.sales.monthly_revenue_report")


Databricks visualization. Run in Databricks to view.